In [1]:
import pickle
import pandas as pd
import numpy as np
import matminer
from matminer.datasets import load_dataset
from pymatgen.core.composition import Composition
from matminer.featurizers.composition import ElementProperty, ElementFraction
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score

In [2]:
# Load models
with open('sc_ep_best_model_yet.pkl', 'rb') as file:
    model_ep = pickle.load(file)
with open('sc_efep_best_model_yet.pkl', 'rb') as file:
    model_efep = pickle.load(file)
with open('sc_ef_best_model_yet.pkl', 'rb') as file:
    model_ef = pickle.load(file)

C:\Users\hp\anaconda3\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.4.2 when using version 1.5.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\hp\anaconda3\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.4.2 when using version 1.5.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [3]:
# Split the dataframe into features and target
def create_composition(formula):
    try:
        return Composition(formula)
    except ValueError:
        print(f"Error parsing formula: {formula}")

def featurize(data):
    data = data.dropna()
    ep_featurizer = ElementProperty.from_preset('magpie')
    ep_ftd = ep_featurizer.featurize_dataframe(data, col_id='_Composition', ignore_errors=True)
    
    # Use ElementFraction instead of ElementProperty
    ef_featurizer = ElementFraction()
    ef_ftd = ef_featurizer.featurize_dataframe(data, col_id='_Composition', ignore_errors=True)
    return ep_ftd, ef_ftd

In [4]:
data = load_dataset("superconductivity2018")['composition'].to_frame().iloc[0:100]

ef_ftd = pd.read_csv("df_sc_ef_ElementFraction_ftd.csv").dropna().reset_index()
ep_ftd = pd.read_csv("df_sc_ep_ElementProperty_Magpie_ftd.csv").dropna().reset_index()

data = data.drop_duplicates()
ef_ftd = ef_ftd.drop_duplicates(subset=["Critical Temp", "_Composition"])
ep_ftd = ep_ftd.drop_duplicates(subset=["Critical Temp", "_Composition"])

Fetching superconductivity2018.json.gz from https://figshare.com/ndownloader/files/31614956 to C:\Users\hp\anaconda3\Lib\site-packages\matminer\datasets\superconductivity2018.json.gz


Fetching https://figshare.com/ndownloader/files/31614956 in MB: 0.18636799999999998MB [00:00,  2.83MB/s]               


In [5]:
ef_ftd

,index,composition,Critical Temp,_Composition,H,He,Li,Be,B,C,...,Pu,Am,Cm,Bk,Cf,Es,Fm,Md,No,Lr
0,0,Ba0.4K0.6Fe2As2,31.20,Ba0.4 K0.6 Fe2 As2,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,Ca0.4Ba1.25La1.25Cu3O6.98,40.10,Ca0.4 Ba1.25 La1.25 Cu3 O6.98,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,Mo0.39Ru0.61,6.90,Mo0.39 Ru0.61,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3,Tm4Os6Sn19,1.10,Tm4 Os6 Sn19,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4,Nd1Bi0.99Pb0.01S2F0.3O0.7,4.85,Nd1 Bi0.99 Pb0.01 S2 F0.3 O0.7,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16401,16401,Al4C3,0.00,Al4 C3,0.0,0.0,0.0,0.0,0.0,0.428571,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
16402,16402,Nb0.96Ta0.04,8.87,Nb0.96 Ta0.04,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
16403,16403,Pb2Sr2Ho0.5Ca0.5Cu2.982Al0.018O8,63.60,Pb2 Sr2 Ho0.5 Ca0.5 Cu2.982 Al0.018 O8,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
16404,16404,Yb0.5Pr0.5Ba2Cu3O6.9,34.80,Yb0.5 Pr0.5 Ba2 Cu3 O6.9,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
data.shape

(10000, 1)

In [7]:
ep_ftd

,index,composition,Critical Temp,_Composition,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,...,MagpieData range GSmagmom,MagpieData mean GSmagmom,MagpieData avg_dev GSmagmom,MagpieData mode GSmagmom,MagpieData minimum SpaceGroupNumber,MagpieData maximum SpaceGroupNumber,MagpieData range SpaceGroupNumber,MagpieData mean SpaceGroupNumber,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber
0,0,Ba0.4K0.6Fe2As2,31.20,Ba0.4 K0.6 Fe2 As2,19.0,56.0,37.0,30.360000,6.214400,26.0,...,2.110663,0.844265,1.013118,0.0,166.0,229.0,63.0,203.800000,30.240000,166.0
1,1,Ca0.4Ba1.25La1.25Cu3O6.98,40.10,Ca0.4 Ba1.25 La1.25 Cu3 O6.98,8.0,57.0,49.0,22.677795,16.074864,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,229.0,217.0,106.949534,102.911141,12.0
2,2,Mo0.39Ru0.61,6.90,Mo0.39 Ru0.61,42.0,44.0,2.0,43.220000,0.951600,44.0,...,0.000000,0.000000,0.000000,0.0,194.0,229.0,35.0,207.650000,16.653000,194.0
3,3,Tm4Os6Sn19,1.10,Tm4 Os6 Sn19,50.0,76.0,26.0,58.000000,10.482759,50.0,...,0.000000,0.000000,0.000000,0.0,141.0,194.0,53.0,159.275862,23.947681,141.0
4,4,Nd1Bi0.99Pb0.01S2F0.3O0.7,4.85,Nd1 Bi0.99 Pb0.01 S2 F0.3 O0.7,8.0,83.0,75.0,36.658000,27.869600,16.0,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,72.206000,49.328776,70.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16401,16401,Al4C3,0.00,Al4 C3,6.0,13.0,7.0,10.000000,3.428571,13.0,...,0.000000,0.000000,0.000000,0.0,194.0,225.0,31.0,211.714286,15.183673,225.0
16402,16402,Nb0.96Ta0.04,8.87,Nb0.96 Ta0.04,41.0,73.0,32.0,42.280000,2.457600,41.0,...,0.000000,0.000000,0.000000,0.0,229.0,229.0,0.0,229.000000,0.000000,229.0
16403,16403,Pb2Sr2Ho0.5Ca0.5Cu2.982Al0.018O8,63.60,Pb2 Sr2 Ho0.5 Ca0.5 Cu2.982 Al0.018 O8,8.0,82.0,74.0,27.138250,19.616202,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,117.531250,105.531250,12.0
16404,16404,Yb0.5Pr0.5Ba2Cu3O6.9,34.80,Yb0.5 Pr0.5 Ba2 Cu3 O6.9,8.0,70.0,62.0,24.705426,17.870921,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,229.0,217.0,110.488372,105.359654,12.0


In [8]:
def record_indices(ep_ftd, ef_ftd, data):
    # Check if composition exists in the original dataset
    data["_Composition"] = data['composition'].apply(create_composition).to_frame()
    compositions = data["composition"].tolist()
    print("Composition:", len(compositions))
    present_indices = []
    missing_indices = []
    
    for idx, composition in enumerate(compositions):
        # Check if the composition exists in the 'composition' column of ep_ftd DataFrame
        if composition in ep_ftd['composition'].tolist():
            # Find the index of the composition in ep_ftd DataFrame
            found_index = ep_ftd[ep_ftd['composition'] == composition]["index"].tolist()
            # Save the found index (or indices) in the present_indices list
            present_indices.extend(found_index)
        else:
            # If the composition is not found, save the enumeration index in the missing_indices list
            missing_indices.append(idx)
    
    print("PRESENT INDICES", present_indices[-10:])
    print("MISSING INDICES", missing_indices)

    # Initialize variables to hold featurized data
    present_ep_ftd = pd.DataFrame()
    present_ef_ftd = pd.DataFrame()
    missing_ep_ftd = pd.DataFrame()
    missing_ef_ftd = pd.DataFrame()

    # If compositions are present, locate them in the featurized data
    if present_indices:
        present_ep_ftd = ep_ftd.loc[ep_ftd["index"].isin(present_indices)]
        present_ef_ftd = ef_ftd.loc[ef_ftd["index"].isin(present_indices)]
     
    # Featurize missing compositions
    if missing_indices:
        missing_data = data.iloc[missing_indices]
        missing_ep_ftd, missing_ef_ftd = featurize(missing_data)
    
    # Combine present and missing featurized data
    final_ep_ftd = pd.concat([present_ep_ftd, missing_ep_ftd], ignore_index=True)
    final_ef_ftd = pd.concat([present_ef_ftd, missing_ef_ftd], ignore_index=True)

    final_nn_ep_ftd = final_ep_ftd.dropna()
    final_nn_ef_ftd = final_ef_ftd.dropna()
    
    final_in_ep_ftd = final_ep_ftd[final_ep_ftd.isnull().any(axis = 1)]["composition"].tolist()
    final_in_ef_ftd = final_ef_ftd[final_ef_ftd.isnull().any(axis = 1)]["composition"].tolist()
    print(final_in_ep_ftd,"\n", final_in_ef_ftd)
    
    return final_nn_ep_ftd, final_nn_ef_ftd, final_in_ep_ftd, final_in_ef_ftd

final_nn_ep_ftd, final_nn_ef_ftd, final_in_ep_ftd, final_in_ef_ftd = record_indices(ep_ftd, ef_ftd, data)

Error parsing formula: Eu1.45Pr0.05Ce0.5Sr2Cu2Nb1O10=z
Error parsing formula: Sm1Ba-1Cu3O6.94
Error parsing formula: Y2C2Br0.5!1.5
Error parsing formula: Hg0.3Pb0.7Sr1.75La0.25Cu1O4+2
Composition: 10000
PRESENT INDICES [9986, 9987, 9988, 9989, 9990, 9991, 9992, 9993, 9994, 9995]
MISSING INDICES [2095, 4554, 7360, 8209, 8650, 9229, 9509, 9806, 9848]


ElementProperty:   0%|          | 0/5 [00:00<?, ?it/s]

ElementFraction:   0%|          | 0/5 [00:00<?, ?it/s]

['Ca1Fe1As1H1', 'La2CuO4', 'La2Sr0Cu0.9Zn0.1O4', 'GePd1', 'Cu1O'] 
 ['Ca1Fe1As1H1', 'La2CuO4', 'La2Sr0Cu0.9Zn0.1O4', 'GePd1', 'Cu1O']


In [9]:
def preprocess_for_model1(ep_ftd):
    # Preprocessing steps for model1
    ep_ftd["having_tc"] = (ep_ftd["Critical Temp"] >= 10).astype(int)
    ep_X = ep_ftd.iloc[:, 4:-1]
    ep_y = ep_ftd['having_tc']
    return ep_X, ep_y

def preprocess_for_model3(ef_ftd):
    # Preprocessing steps for model3
    ef_ftd["having_tc"] = (ef_ftd["Critical Temp"] >= 10).astype(int)

    ef_X = ef_ftd.iloc[:, 4:-1]
    ef_y = ef_ftd['having_tc']
    return ef_X, ef_y

def preprocess_for_model2(ep_ftd, ef_ftd):
    ef_ftd = ef_ftd.iloc[:, 2:]
    ep_ftd = ep_ftd.iloc[:, 2:]
    
    print(ef_ftd.shape)
    print(ep_ftd.shape)
    
    merged_df = pd.merge(ef_ftd, ep_ftd, left_on=["Critical Temp", "_Composition"], right_on=["Critical Temp", "_Composition"], how="inner")
    merged_df = pd.merge(ef_ftd, ep_ftd, left_on=["Critical Temp", "_Composition"], right_on=["Critical Temp", "_Composition"], how="inner")
    merged_df["having_tc"] = (merged_df["Critical Temp"] >= 10).astype(int)

    efep_X = merged_df.iloc[:, 2:-1]
    efep_y = merged_df['having_tc']    
    
    print(efep_X.shape)
    return efep_X, efep_y, merged_df

In [10]:
def ensemble_predict(ep_ftd, ef_ftd):
    # Preprocess the sample for each model
    ep_X, ep_y = preprocess_for_model1(ep_ftd)
    efep_X, efep_y, merged_df = preprocess_for_model2(ep_ftd, ef_ftd)
    ef_X, ef_y = preprocess_for_model3(ef_ftd)
    
    pred_ep = model_ep.predict(ep_X)
    pred_efep = model_efep.predict(efep_X)
    pred_ef = model_ef.predict(ef_X)
    
    # Combine predictions using majority vote
    n_samples = len(pred_ep)
    ensemble_pred = np.zeros(n_samples)
    for i in range(n_samples):
        # Count votes for each class
        class_counts = np.bincount([pred_ep[i], pred_efep[i], pred_ef[i]])
        # Majority class is the one with the highest count
        ensemble_pred[i] = np.argmax(class_counts)

    ensemble_pred = ensemble_pred.astype(int)
    
    return pred_ep, pred_efep, pred_ef, ensemble_pred, ep_y, efep_y, ef_y

pred_ep, pred_efep, pred_ef, ensemble_pred, ep_y, efep_y, ef_y = ensemble_predict(final_nn_ep_ftd, final_nn_ef_ftd)


(9991, 105)
(9991, 135)
(9991, 235)


In [11]:
# Assuming 'y_true' contains the true labels and 'y_pred' contains the predictions from your ensemble
accuracy = accuracy_score(efep_y, ensemble_pred)
print("Accuracy:", accuracy)

# Confusion Matrix
conf_matrix = confusion_matrix(ep_y, ensemble_pred)
print("Confusion Matrix:\n", conf_matrix)

# Precision
precision = precision_score(ep_y, ensemble_pred)
print("Precision:", precision)

# Recall
recall = recall_score(efep_y, ensemble_pred)
print("Recall:", recall)

# F1 Score
f1 = f1_score(ef_y, ensemble_pred)
print("F1 Score:", f1)

# ROC-AUC Score
roc_auc = roc_auc_score(ef_y, ensemble_pred)
print("ROC-AUC Score:", roc_auc)

Accuracy: 0.9632669402462216
Confusion Matrix:
 [[5941  279]
 [  88 3683]]
Precision: 0.9295810196870268
Recall: 0.9766640148501724
F1 Score: 0.9525410578042157
ROC-AUC Score: 0.9659043546919672


In [12]:
# Accuracy: 0.9748884951426651
# Confusion Matrix:
#  [[9836  284]
#  [ 127 6120]]
# Precision: 0.9556527170518426
# Recall: 0.9796702417160237
# F1 Score: 0.9675124496087266
# ROC-AUC Score: 0.9758035003046521

In [13]:
# Accuracy: 0.9747343092039302
# Confusion Matrix:
#  [[6026  178]
#  [  74 3696]]
# Precision: 0.9540526587506454
# Recall: 0.9803713527851459
# F1 Score: 0.967032967032967
# ROC-AUC Score: 0.9758400928980533